In [45]:
from langgraph.graph import StateGraph, START, END

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
#from langchain_opneai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os


In [46]:
load_dotenv()

True

In [47]:
#for oppenai
#model = ChatOpenAI()

In [48]:
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    max_new_tokens=256
)

model = ChatHuggingFace(llm=llm)

In [49]:
#create a state
class LLState(TypedDict):
    question:str
    answer :str


In [50]:
def llm_qa(state:LLState) -> LLState:
    #extract the question from the state
    question = state['question']

    #form a prompt
    prompt = f'Answer the following question: {question}'
    # asked the question to the llm
    answer= model.invoke(prompt)

    #update the state with the answer
    state['answer'] = answer

    return state

In [51]:
#create our graph
graph = StateGraph(LLState)

# crete Node
graph.add_node('llm_qa',llm_qa)

# create edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compile the graph
workflow =graph.compile()


In [57]:
#execute the wrokflow
initial_state = {'question': 'What is the capital of France?'}
final_state = workflow.invoke(initial_state)
print(final_state['answer'])

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6b015f7d-2faacdd10c61d73e2d2f8b1c;4702e0ff-6085-43cb-ba78-a95a607e178c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.